%md
##### Note: block 5 contains both the enforcement  (expected to fail, caught by the try/except) and the evolution fix (runs right after, with mergeSchema=true). If you re-run the whole notebook top to bottom, block 3 and block 5 will duplicate rows (both use append without dedup) — fine for this demo, but worth revisiting with a MERGE on order_id+offset if you want the notebook to be safely re-runnable later.

In [ ]:
%run ../setup/config

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# read bronze  and parse the base event (without discount_code)

df_bronze_orders = spark.table(BRONZE_ORDER_FULL_TABLE)

df_parsed = df_bronze_orders.withColumn(
    "event",
    F.from_json(F.col("raw_json"), ORDER_EVENT_SCHEMA)
)

df_silver_orders = (
    df_parsed
    .select(
        F.col("event.order_id").alias("order_id"),
        F.col("event.item_id").alias("item_id"),
        F.col("event.event_timestamp").cast("timestamp").alias("event_timestamp"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.current_timestamp().alias("_ingestion_timestamp"),
    )
)

# write the normal events (no discount_code) to Silver

(
    df_silver_orders.write
    .format("delta")
    .mode("append")
    .saveAsTable(SILVER_ORDER_FULL_TABLE)
)


In [ ]:
# isolate the schema_change scenario events (they carry discount_code  a field the table doesn't have yet)

df_bronze_schema_change = (
    spark.table(BRONZE_ORDER_FULL_TABLE)
    .filter(F.col("raw_json").contains("discount_code"))
)

df_parsed_v2 = df_bronze_schema_change.withColumn(
    "event",
    F.from_json(F.col("raw_json"), ORDER_EVENT_SCHEMA_V2)
)

df_silver_orders_v2 = (
    df_parsed_v2
    .select(
        F.col("event.order_id").alias("order_id"),
        F.col("event.item_id").alias("item_id"),
        F.col("event.event_timestamp").cast("timestamp").alias("event_timestamp"),
        F.col("event.discount_code").alias("discount_code"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.current_timestamp().alias("_ingestion_timestamp"),
    )
)

In [ ]:
# S
# schema enforcement demo: write without mergeSchema (expected to fail with DELTA_METADATA_MISMATCH)

try:
    (
        df_silver_orders_v2.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_ORDER_FULL_TABLE)
    )
except Exception as e:
    print(f"Expected schema enforcement error:\n{e}")

# schema evolution demo: retry with mergeSchema=true (Delta widens the table automatically)

(
    df_silver_orders_v2.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_ORDER_FULL_TABLE)
)

print(f"{df_silver_orders_v2.count()} rows written to {SILVER_ORDER_FULL_TABLE} (with schema evolution)")